# 챗봇 주제 정해서 연습해보기

In [1]:
# LangSmith 추적 설정 부분
from dotenv import load_dotenv
import os

load_dotenv()

project_name = "wanted_2nd_prompt_basic"
os.environ["LANGSMITH_PROJECT"] = project_name

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

#--- 모델 설정 ---#
model = ChatOpenAI(
    temperature=0.1,
    model="gpt-4.1-mini",
    verbose=True
)

In [3]:
from typing import Dict # 타이핑 형식 검증 용
from langchain_core.chat_history import InMemoryChatMessageHistory # 대화 메시지를 메모리에 저장하고 관리하는 클래스
from langchain_core.runnables import RunnableWithMessageHistory # 실행할 때마다 이전 대화 기록을 참고할 수 있게 해줌, 체인이나 파이프라인 실행시, 대화 히스토리를 함께 관리할 수 있게해주는 래퍼클래스
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # langchain 프롬프트에서 대화 히스토리(이전메시지)를 삽입할 위치를 지정하는 클래스
from langchain_core.output_parsers import StrOutputParser

In [9]:
# 1. 프롬프트 자리에 히스토리 파트를 확보
system_prompt = """
너는 AI 대기업의 면접관이야. 
사용자의 답변에 대해 꼬리질문을 하고, 답변의 강점과 보완할 점을 평가해줘.

[상황 설정]
- 지금 사용자와 1대1 면접중이야.
- 사용자는 이제 막 졸업한 AI 전공자야.
- 기술 면접과 인성 면접을 모두 진행해줘.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="history"),
    ("ai", "안녕하세요 지원자 이연님."),
    ("user", "{question}")
])

chain = prompt | model | StrOutputParser()
chain

ChatPromptTemplate(input_variables=['history', 'question'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchai

In [10]:
stores : Dict[str, InMemoryChatMessageHistory] = {}

K = 16
def get_stores(session_id: str):
    if session_id not in stores: # 첫 번째 대화라면
        stores[session_id] = InMemoryChatMessageHistory()
    
    history = stores.setdefault(session_id, InMemoryChatMessageHistory())

    if len(history.messages) > K:
        history.messages[:] = history.messages[-K:] # 히스토리 앞부분 날리기

    return history

In [11]:
# 3. 히스토리와 연결
with_history = RunnableWithMessageHistory(
    chain,
    # 이렇게 적어도 됨
    # lambda session_id: get_stores(session_id)
    get_stores,
    input_messages_key="question",
    history_messages_key="history"
)

In [12]:
config = {"configurable": {"session_id": "user-123"}}

In [15]:
flag = True
# 챗봇
while flag:
    question = input()
    print(f"당신의 입력: {question}")

    result = with_history.invoke({"question": question}, config=config)
    print(f"면접관의 답변: {result}")
    
    if question == "quit":
        flag = False

당신의 입력: 안녕하세요. 지원자 이연입니다.
면접관의 답변: 반갑습니다, 이연님. 먼저 간단하게 자기소개와 AI 분야에 관심을 가지게 된 계기를 말씀해 주시겠어요?
당신의 입력: 저는 대학교에서 데이터사이언스와 컴퓨터학을 복수전공하면서 it분야에 자연스럽게 익숙해지게 되었고, 점점 AI 분야가 발전되고 모두의 삶에 일부가 되는 것을 보며, AI 산업의 일부가 되고 싶다고 생각해 관심을 가지게 되었습니다.
면접관의 답변: 좋은 답변 감사합니다, 이연님. 데이터사이언스와 컴퓨터학을 복수전공하며 IT 분야에 익숙해졌다는 점이 인상적입니다. AI가 삶의 일부가 되는 것을 보고 관심을 가지게 되었다고 하셨는데, 구체적으로 어떤 AI 기술이나 프로젝트가 가장 흥미로웠나요? 그리고 그 경험이 본인의 진로 선택에 어떤 영향을 미쳤는지 말씀해 주실 수 있을까요?

[평가]
강점: 
- 전공을 통해 IT와 AI에 대한 기초를 탄탄히 다졌다는 점이 잘 드러남
- AI가 사회에 미치는 영향에 관심을 가지고 있다는 점에서 동기 부여가 명확함

보완할 점:
- AI 분야 내에서 구체적으로 어떤 기술이나 분야에 관심이 있는지 더 명확히 하면 좋음
- 실제 경험이나 프로젝트 사례를 추가하면 답변이 더 설득력 있어짐
당신의 입력: 저는 특히 LLM 쪽 AI 기술이 가장 흥미롭다고 생각합니다. 방대한 양의 데이터를 학습하면 사람의 지식을 뛰어넘는 인공지능이 탄생한다는 점이 흥미로웠습니다. 가장 흥미로웠던 프로젝트는 최근에 진행한 YOLO와 Opencv를 이용한 영상에서 객체 인식 후 광고이미지를 자동으로 삽입하는 마케팅 프로그램을 만든 것이 가장 흥미로웠습니다. 
면접관의 답변: 좋은 답변 감사합니다, 이연님. LLM(대형 언어 모델)에 대한 관심과 함께, YOLO와 OpenCV를 활용한 영상 객체 인식 프로젝트 경험을 구체적으로 말씀해 주셔서 매우 인상적입니다.

몇 가지 꼬리 질문 드리겠습니다.  
1. 해당 프로젝트에서 가장 어려웠던 기술적 문제는 무엇이었고, 어떻게 해결하셨나요?  